# Mission 4: Follow the Trail

Track a red candidate through controlled synthetic corridor clips and test whether its observed center crosses the exit line. The generated silhouette is not a real volunteer, and the tracker does not identify a carrier by face or name.

## Objectives and setup
Use VideoCapture, inspect actual metadata, process frames, keep recent centroids, mark a line crossing and save annotated output. From repository root: `python -m pip install -e ".[test]"`; run Jupyter Lab. Choose the steady, fast, or temporary-occlusion/distractor clip in `assets/videos/`. `VideoCapture` reads frames; `VideoWriter` exports an annotated MP4.

Exercise: predict what happens to the trail when no red candidate is found.

In [ ]:
from pathlib import Path
import sys, cv2
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from the_last_chai.video_tracking import track_video
video_path = ROOT / 'assets/videos/corridor_staged.mp4'
capture = cv2.VideoCapture(str(video_path))
metadata = {'fps': capture.get(cv2.CAP_PROP_FPS), 'frames': capture.get(cv2.CAP_PROP_FRAME_COUNT), 'width': capture.get(cv2.CAP_PROP_FRAME_WIDTH), 'height': capture.get(cv2.CAP_PROP_FRAME_HEIGHT)}
ok, first = capture.read(); capture.release()
print(metadata, 'first frame available:', ok)
plt.figure(figsize=(10,5)); plt.imshow(cv2.cvtColor(first, cv2.COLOR_BGR2RGB)); plt.axis('off');

For each frame, the shared tracker converts to HSV, cleans the red mask, filters tiny contours, selects the largest remaining candidate, and records its center. It preserves only recent valid points; a missing frame resets the crossing comparison so gaps are not treated as motion. Crossing is reported only for observed points on both sides. `min_area` and `line_x` are parameters.

Exercise: move `line_x` and see whether the measured crossing frame changes. The video metadata, not a hard-coded count, determines how many frames are processed.

In [ ]:
result = track_video(str(video_path), str(ROOT / 'outputs/level4_annotated.mp4'), line_x=480, min_area=700)
result

In [ ]:
# TODO: Move the exit line to x=420 or x=520. Compare crossing time or no-crossing.
line_trial = track_video(str(ROOT / 'assets/videos/corridor_fast.mp4'), line_x=420, min_area=700)
print('Your alternate clip/line result:', line_trial['crossing'])

Hint 1: verify `capture.isOpened()` and the FPS. Hint 2: use a line x-coordinate inside the 640-pixel frame. Hint 3: inspect the red box/trail and crossing frame in the output.

Clue: the observed red candidate crosses; **the prepared fictional system log** separately says `23:44: CORRUPTION SCRIPT EXECUTED`. The story case file names Kabir as carrier; the synthetic tracker does not identify a person. Challenge: what happens when an occluder hides the bag? Common errors: wrong codec, output writer not opened, dividing frame index by zero FPS, or claiming crossing after a missing detection.

In [ ]:
output = cv2.VideoCapture(str(ROOT / 'outputs/level4_annotated.mp4'))
ok, frame = output.read(); output.release()
assert ok, 'Annotated output did not reopen; check the local codec support.'
plt.figure(figsize=(10,5)); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off');

## Recap
The tracker identifies a red region, estimates its center and observes a crossing. Identity, ownership, incident narrative and the 23:44 log come from separate fictional case materials. Computer vision supports the story but is not forensic proof.